Set up, import pyspark

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

Create Spark session, use partitions

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Step 1: groupBy/agg - yearly audio feature averages
yearly_features = (
    data
    .filter(
        (F.col("year").isNotNull()) &
        (F.col("danceability").cast("double").between(0, 1)) &
        (F.col("energy").cast("double").between(0, 1)) &
        (F.col("valence").cast("double").between(0, 1))
    )
    .groupBy("year")
    .agg(
        F.count("*").alias("track_count"),
        F.round(F.avg(F.col("danceability").cast("double")), 3).alias("avg_danceability"),
        F.round(F.avg(F.col("energy").cast("double")), 3).alias("avg_energy"),
        F.round(F.avg(F.col("valence").cast("double")), 3).alias("avg_valence"),
        F.round(F.avg(F.col("tempo").cast("double")), 3).alias("avg_tempo")
    )
)

# Step 2: groupBy/agg - explicit rate per year
explicit_by_year = (
    data
    .filter(F.col("year").isNotNull())
    .groupBy("year")
    .agg(
        F.round(
            F.sum(F.when(F.col("explicit") == "True", 1).otherwise(0)) /
            F.count("*"), 3
        ).alias("explicit_rate")
    )
)

# Step 3: join - combine both tables on year
trend_table = yearly_features.join(explicit_by_year, on="year", how="inner")

# Step 4: window - add year-over-year change in avg danceability
window_spec = Window.orderBy(F.col("year").cast("int"))

trend_table_final = (
    trend_table
    .withColumn(
        "yoy_danceability_change",
        F.round(
            F.col("avg_danceability") - F.lag("avg_danceability", 1).over(window_spec),
            3
        )
    )
    .orderBy(F.col("year").cast("int"))
)

print("Music trend table — ready for final project analysis:")
trend_table_final.show(20)

In [ ]:
from pyspark.sql import SparkSession
import sys

spark = (
    SparkSession.builder
    .appName("Team9_Sec2_Sprint5")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# Version checks
print("Python version:", sys.version)
print("Spark version: ", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
!java -version

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Spark version:  4.0.2
Shuffle partitions: 8
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:

import json
from pathlib import Path

#install kaggle
!pip install kaggle --quiet

#Upload your kaggle key here
from google.colab import files
uploaded = files.upload()

if len(uploaded) == 0 | len(uploaded) > 1:
  raise ValueError("Only upload one file")


for filename, content in uploaded.items():
  try:
    data = json.loads(content.decode("utf-8"))
    print("JSON uploaded")

  except json.JSONDecodeError as e:
    print("ERROR: Needs valid kaggle json key")




In [ ]:
#Move the file and set permissions

filename = list(uploaded.keys())[0]

!mkdir -p ~/.kaggle


!cp "{filename}" ~/.kaggle/
!chmod 600 ~/.kaggle/"{filename}"
!cp ~/.kaggle/"{filename}" ~/.kaggle/kaggle.json
!ls -ltr ~/.kaggle




In [ ]:
!kaggle datasets list -s "tracks_features.csv"

In [ ]:
!kaggle datasets download -d 'rodolfofigueroa/spotify-12m-songs' -p ../data/dataset

!unzip ../data/dataset/spotify-12m-songs.zip -d ../data/dataset


In [ ]:

from pyspark.sql.functions import col


data = spark.read.csv('/data/dataset/tracks_features.csv', header=True, inferSchema=True, sep=",")


data.printSchema()
print(data.columns)
data.show(10)


In [ ]:
!ls -ltr ../data/dataset
!chmod 777 ../data/dataset/tracks_features.csv

Top 20 artists by amount of **songs**

In [ ]:


from pyspark.sql.functions import rand


top_artists = (
    data.select("artists")
    .groupBy("artists")
    .count()
    .orderBy("count", ascending=False)

)


top_artists.show(20, truncate=False)

Frequency Table: Song count by year


In [ ]:
from pyspark.sql.functions import col, avg, round as spark_round

# The raw CSV stores year as a string and sometimes as a float like '4.0'.
# We cast to double first (handles decimal strings), then to int to get
# clean integer years, then filter out junk values outside 1900-2025.
year_count = (
    data
    .withColumn("year_clean", col("year").cast("double").cast("int"))
    .filter((col("year_clean") >= 1900) & (col("year_clean") <= 2025))
    .groupBy("year_clean")
    .count()
    .orderBy("year_clean")
)

year_count.show(30)
year_count.printSchema()

Average Audio Features by Year

In [ ]:
from pyspark.sql.functions import col, avg, round as spark_round

# Compute average audio features per year.
# Useful for the final project to analyze how music characteristics
# like danceability, energy, and mood (valence) have shifted over time.
yearly_features = (
    data
    .withColumn("year_clean",   col("year").cast("double").cast("int"))
    .withColumn("danceability", col("danceability").cast("double"))
    .withColumn("energy",       col("energy").cast("double"))
    .withColumn("valence",      col("valence").cast("double"))
    .filter((col("year_clean") >= 1950) & (col("year_clean") <= 2020))
    .groupBy("year_clean")
    .agg(
        spark_round(avg("danceability"), 3).alias("avg_danceability"),
        spark_round(avg("energy"),       3).alias("avg_energy"),
        spark_round(avg("valence"),      3).alias("avg_valence")
    )
    .orderBy("year_clean")
)

yearly_features.show(20)